# Working with records: grouping, counting and deduplicating

Data often arrives as a list of dictionaries, one per record. Grouping uses a dictionary of lists, counting uses a dictionary of numbers or collections.Counter, sorting uses sorted with a key function, and "latest per key" keeps one record per key by comparing timestamps. These are the same patterns as GROUP BY and ROW_NUMBER in SQL.

**How to run in Google Colab:** run the cells from top to bottom (Runtime → Run all). Each part asks you to predict the result first; the expected results are listed at the end. For Databricks features such as Unity Catalog tables, use the Databricks Free Edition version of this lab.

From the Learning Hub lesson: [Working with records: grouping, counting and deduplicating](https://www.datapreplab.com/hub#topic/py-records) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** group, count, sort and deduplicate the same records.

**Part A: data.**

In [ ]:
rows = [{"id": 1, "dept": "Data", "salary": 60000, "updated": 2},
        {"id": 2, "dept": "Data", "salary": 40000, "updated": 1},
        {"id": 1, "dept": "Data", "salary": 65000, "updated": 5},
        {"id": 3, "dept": "Ops",  "salary": 50000, "updated": 3}]

**Part B: count and sort.**

In [ ]:
from collections import Counter
print(Counter(r["dept"] for r in rows))                      # Data 3, Ops 1 (before dedup)
print([r["salary"] for r in sorted(rows, key=lambda r: r["salary"], reverse=True)])
# [65000, 60000, 50000, 40000]

**Part C: deduplicate, then aggregate.**

In [ ]:
latest = {}
for r in rows:
    current = latest.get(r["id"])
    if current is None or r["updated"] > current["updated"]:
        latest[r["id"]] = r

groups = {}
for r in latest.values():
    groups.setdefault(r["dept"], []).append(r["salary"])

for dept, salaries in groups.items():
    print(dept, len(salaries), sum(salaries) / len(salaries))

**Part D: see the bug.** Compute the Data average **before** deduplicating:

In [ ]:
data = [r["salary"] for r in rows if r["dept"] == "Data"]
print(sum(data) / len(data))                                 # 55000.0, wrong

Expected evidence:

```text
Part B -> Counter({'Data': 3, 'Ops': 1}) ; [65000, 60000, 50000, 40000]
Part C -> Data 2 52500.0 ; Ops 1 50000.0
Part D -> 55000.0 (employee 1 counted twice)
```